In [3]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.ensemble import RandomForestClassifier
import pandas as pd
from scipy.fft import fft
import numpy as np

In [4]:
from scipy.signal import butter, lfilter

def butter_bandpass(lowcut, highcut, fs, order=5):
    nyq = 0.5 * fs
    low = lowcut / nyq
    high = highcut / nyq
    b, a = butter(order, [low, high], btype='band')
    return b, a


def butter_bandpass_filter(data, lowcut, highcut, fs, order=5):
    b, a = butter_bandpass(lowcut, highcut, fs, order=order)
    y = lfilter(b, a, data)
    return y

def signal_frequency_band_energies(sampled_signal, frequency_bands, sampling_frequency,
order=5):
    energies = []
    for bands in frequency_bands:
        energies.append(np.sum(np.abs(butter_bandpass_filter(sampled_signal, bands[0], bands[1],
        sampling_frequency, order))**2))
    return energies

In [5]:
data = pd.read_csv("./dataSet.csv")#get data from csv
group=data.groupby(['subject','label'])
print(group.size())

subject  label
0        0        2142701
         1         800800
         2         430500
         3         253400
         4         537599
                   ...   
14       3         260400
         4         511700
         5          40600
         6          41299
         7          39900
Length: 119, dtype: int64


In [6]:
keys=group.groups.keys()
print(keys)

dict_keys([(0, 0), (0, 1), (0, 2), (0, 3), (0, 4), (0, 6), (0, 7), (1, 0), (1, 1), (1, 2), (1, 3), (1, 4), (1, 5), (1, 6), (1, 7), (2, 0), (2, 1), (2, 2), (2, 3), (2, 4), (2, 5), (2, 6), (2, 7), (3, 0), (3, 1), (3, 2), (3, 3), (3, 4), (3, 5), (3, 6), (3, 7), (4, 0), (4, 1), (4, 2), (4, 3), (4, 4), (4, 5), (4, 6), (4, 7), (5, 0), (5, 1), (5, 2), (5, 3), (5, 4), (5, 5), (5, 6), (5, 7), (6, 0), (6, 1), (6, 2), (6, 3), (6, 4), (6, 5), (6, 6), (6, 7), (7, 0), (7, 1), (7, 2), (7, 3), (7, 4), (7, 5), (7, 6), (7, 7), (8, 0), (8, 1), (8, 2), (8, 3), (8, 4), (8, 5), (8, 6), (8, 7), (9, 0), (9, 1), (9, 2), (9, 3), (9, 4), (9, 5), (9, 6), (9, 7), (10, 0), (10, 1), (10, 2), (10, 3), (10, 4), (10, 5), (10, 6), (10, 7), (11, 0), (11, 1), (11, 2), (11, 3), (11, 4), (11, 5), (11, 6), (11, 7), (12, 0), (12, 1), (12, 2), (12, 3), (12, 4), (12, 5), (12, 6), (12, 7), (13, 0), (13, 1), (13, 2), (13, 3), (13, 4), (13, 5), (13, 6), (13, 7), (14, 0), (14, 1), (14, 2), (14, 3), (14, 4), (14, 5), (14, 6), (14, 7

In [7]:
windows_size=64
process_data=[]
for i in keys:
    print(i)
    temp=group.get_group(i).values.tolist()
    print(len(temp))
    begin=0
    end=windows_size
    while end<len(temp):
        temp_process_data=[]
        temp_data=np.array(temp[begin:end]).T
        for x in range(1,4):#ACC0,ACC1,ACC2
            temp_process_data.append(temp_data[x].mean())
        for x in range(5,9):#EMG,EDA,resp,temp
            temp_process_data.append(temp_data[x].mean())
        #ECG_ULF,ECG_LF,ECG_HF,ECG_UHF
        for x in signal_frequency_band_energies(fft(temp_data[4]),[[0.01, 0.04], [0.04, 0.15], [0.15, 0.4], [0.4, 1.0]],700):
            temp_process_data.append(x)
        temp_process_data.append(temp_data[9][0])#label
        temp_process_data.append(temp_data[10][0])#subject
        #print(temp_process_data)
        process_data.append(temp_process_data)
        begin+=windows_size
        end+=windows_size


(0, 0)
2142701
(0, 1)
800800
(0, 2)
430500
(0, 3)
253400
(0, 4)
537599
(0, 6)
45500
(0, 7)
44800
(1, 0)
2345699
(1, 1)
798000
(1, 2)
448000
(1, 3)
262500
(1, 4)
546001
(1, 5)
51100
(1, 6)
46900
(1, 7)
46900
(2, 0)
2314199
(2, 1)
810601
(2, 2)
444500
(2, 3)
260400
(2, 4)
563500
(2, 5)
35699
(2, 6)
30800
(2, 7)
36401
(3, 0)
2142700
(3, 1)
838600
(3, 2)
451500
(3, 3)
261800
(3, 4)
555800
(3, 5)
50401
(3, 6)
30799
(3, 7)
49000
(4, 0)
2733499
(4, 1)
826000
(4, 2)
455000
(4, 3)
260400
(4, 4)
550900
(4, 5)
40600
(4, 6)
35001
(4, 7)
48300
(5, 0)
1472098
(5, 1)
830200
(5, 2)
448000
(5, 3)
260401
(5, 4)
553001
(5, 5)
35000
(5, 6)
30799
(5, 7)
37101
(6, 0)
1616300
(6, 1)
818300
(6, 2)
469000
(6, 3)
258999
(6, 4)
557200
(6, 5)
34300
(6, 6)
35701
(6, 7)
36400
(7, 0)
1435700
(7, 1)
826000
(7, 2)
451500
(7, 3)
260400
(7, 4)
555100
(7, 5)
42000
(7, 6)
43400
(7, 7)
42000
(8, 0)
1589000
(8, 1)
826000
(8, 2)
507500
(8, 3)
260400
(8, 4)
557200
(8, 5)
35700
(8, 6)
31500
(8, 7)
39900
(9, 0)
1443400
(9, 1)
8

In [8]:
process_data=pd.DataFrame(process_data)
process_data=process_data.rename(columns={0:"ACC0",
                                          1:"ACC1",
                                          2:"ACC2",
                                          3:"EMG",
                                          4:"EDA",
                                          5:"resp",
                                          6:"temp",
                                          7:"ECG_ULF",
                                          8:"ECG_LF",
                                          9:"ECG_HF",
                                          10:"ECG_UHF",
                                          11:"label",
                                          12:"subject"})
process_data.to_csv("process_dataSet.csv")